# 개별종목 조합C — LightGBM

`기본모델/04.LightGBM.ipynb`과 같은 `models.lightgbm.build_lightgbm_baseline`을 가져오고
조합C 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.lightgbm import build_lightgbm_baseline  # noqa: E402

MODEL_NAME = 'LightGBM'
MODEL_BUILDER = build_lightgbm_baseline


In [2]:
# 2. 조합C의 피처 값만 지정합니다.
import json

COMBINATION = 'C'
FEATURE_COLUMNS = (
    'ret_5',
    'overnight_gap',
    'intraday_return',
    'close_location',
    'bb_position',
    'rsi_14',
    'volume_z_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합C 피처: ('ret_5', 'overnight_gap', 'intraday_return', 'close_location', 'bb_position', 'rsi_14', 'volume_z_20')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4431,0.5012,-0.0581,0.3554,0.3623,0.0493,0.3743,0.2255,0.3156
1,2,balanced,980,20150123,20150421,0.3630,0.3978,-0.0348,0.3392,0.3437,0.0188,0.3552,0.2325,0.2999
2,3,balanced,1210,20151228,20160328,0.3625,0.3762,-0.0137,0.3581,0.3580,0.0400,0.3571,0.3157,0.3441
3,4,balanced,1439,20161202,20170228,0.4136,0.4617,-0.0481,0.3617,0.3652,0.0567,0.3719,0.2401,0.3209
4,5,balanced,1669,20171113,20180207,0.3794,0.3901,-0.0107,0.3665,0.3680,0.0535,0.3797,0.3025,0.3460
5,6,balanced,1899,20181024,20190118,0.3952,0.3725,0.0227,0.3939,0.3945,0.0938,0.4058,0.3753,0.3879
6,7,balanced,2129,20190930,20191224,0.4184,0.4781,-0.0598,0.3637,0.3661,0.0550,0.3776,0.2471,0.3266
7,8,balanced,2359,20200902,20201130,0.3823,0.3476,0.0347,0.3807,0.3826,0.0747,0.3763,0.3800,0.3810
8,9,balanced,2589,20210806,20211105,0.3612,0.3914,-0.0302,0.3511,0.3705,0.0421,0.3676,0.2146,0.2919
9,10,balanced,2818,20220714,20221012,0.3482,0.3454,0.0028,0.3436,0.3512,0.0295,0.3559,0.2484,0.3059


,OOS 폴드 평균
accuracy,0.3874
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0094
macro_f1,0.3653
balanced_accuracy,0.3697
mcc,0.0567
pr_auc_macro_ovr,0.3741
down_recall,0.2870
core_harmonic_mean,0.3378


재실행 명령: python scripts/run_stock_model_experiment.py
